# Map-Aware Multi-Modal Motion Forecasting on WOMD

MTR-Lite: agent history + HD map lanes + social attention + K=6 trajectory modes.
Trained on WOMD v1.3.1 validation shards, evaluated with minADE@6 across tail slices.

In [ ]:
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q waymo-open-dataset-tf-2-12-0 --no-deps
!pip install -q tensorflow numpy google-cloud-storage tqdm

import io
import os
import random

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation, FFMpegWriter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import tensorflow as tf
from waymo_open_dataset.protos import scenario_pb2

from google.colab import auth
from google.cloud import storage
from IPython.display import HTML, FileLink, display
from tqdm import tqdm

auth.authenticate_user()

PROJECT_ID  = 'womd-imitation'
GCS_BUCKET  = 'womd-imitation-data'
WOMD_BUCKET = 'waymo_open_dataset_motion_v_1_3_1'

_gcs        = storage.Client(project=PROJECT_ID)
bucket      = _gcs.bucket(GCS_BUCKET)
womd_bucket = _gcs.bucket(WOMD_BUCKET)

val_blobs  = list(womd_bucket.list_blobs(prefix='uncompressed/scenario/validation/'))
val_shards = [b for b in val_blobs if 'tfrecord' in b.name]
map_blobs  = list(bucket.list_blobs(prefix='features/map_v1/'))

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'torch  : {torch.__version__}  cuda={torch.cuda.is_available()}')
print(f'tf     : {tf.__version__}')
print(f'device : {device}')
print(f'WOMD val shards    : {len(val_shards)}')
print(f'map feature shards : {len(map_blobs)}')


## Feature extraction

Each scenario yields per-agent samples with three input streams:
agent history (position, heading, velocity), nearby lane polylines from the HD map,
and the histories of up to 8 surrounding agents.

In [ ]:
HIST_STEPS   = 11
FUTURE_STEPS = 80
MAX_LANES    = 32
MAX_PTS_LANE = 20
MAX_AGENTS   = 8

AGENT_TYPE_INT = {
    scenario_pb2.Track.TYPE_VEHICLE    : 0,
    scenario_pb2.Track.TYPE_PEDESTRIAN : 1,
    scenario_pb2.Track.TYPE_CYCLIST    : 2,
    scenario_pb2.Track.TYPE_OTHER      : 3,
    scenario_pb2.Track.TYPE_UNSET      : 3,
}


def resample_polyline(pts, n=20):
    if len(pts) == 0:
        return np.zeros((n, 2), dtype=np.float32)
    pts = np.array(pts, dtype=np.float32)
    if len(pts) == 1:
        return np.tile(pts, (n, 1))
    diffs   = np.diff(pts, axis=0)
    dists   = np.sqrt((diffs**2).sum(axis=1))
    cumdist = np.concatenate([[0], np.cumsum(dists)])
    total   = cumdist[-1]
    if total < 1e-6:
        return np.tile(pts[0], (n, 1))
    resampled = np.zeros((n, 2), dtype=np.float32)
    for i, d in enumerate(np.linspace(0, total, n)):
        idx     = np.clip(np.searchsorted(cumdist, d, side='right') - 1, 0, len(pts) - 2)
        seg_len = cumdist[idx + 1] - cumdist[idx]
        t       = (d - cumdist[idx]) / (seg_len + 1e-6)
        resampled[i] = pts[idx] + t * (pts[idx + 1] - pts[idx])
    return resampled


def extract_map_features(scenario, origin, heading, max_lanes=MAX_LANES, n_pts=MAX_PTS_LANE):
    cos_h, sin_h = np.cos(-heading), np.sin(-heading)
    R = np.array([[cos_h, -sin_h], [sin_h, cos_h]], dtype=np.float32)

    lane_pts   = np.zeros((max_lanes, n_pts, 2), dtype=np.float32)
    lane_valid = np.zeros((max_lanes,),           dtype=bool)
    lane_type  = np.zeros((max_lanes,),           dtype=np.int32)

    lanes = []
    for feat in scenario.map_features:
        if feat.HasField('lane'):
            pts = [(p.x, p.y) for p in feat.lane.polyline]
            if len(pts) < 2:
                continue
            dist = ((pts[0][0] - origin[0])**2 + (pts[0][1] - origin[1])**2)**0.5
            lanes.append((dist, pts, int(feat.lane.type)))

    lanes.sort(key=lambda x: x[0])
    for i, (_, pts, ltype) in enumerate(lanes[:max_lanes]):
        resampled     = resample_polyline(pts, n_pts)
        lane_pts[i]   = (resampled - origin) @ R.T
        lane_valid[i] = True
        lane_type[i]  = ltype

    return lane_pts, lane_valid, lane_type


def extract_social_agents(scenario, focal_idx, origin, heading, max_agents=MAX_AGENTS):
    cos_h, sin_h = np.cos(-heading), np.sin(-heading)
    R = np.array([[cos_h, -sin_h], [sin_h, cos_h]], dtype=np.float32)

    social_xy    = np.zeros((max_agents, HIST_STEPS, 2), dtype=np.float32)
    social_valid = np.zeros((max_agents, HIST_STEPS),    dtype=bool)
    social_mask  = np.zeros((max_agents,),               dtype=bool)

    candidates = []
    for i, track in enumerate(scenario.tracks):
        if i == focal_idx:
            continue
        states = track.states[:HIST_STEPS]
        valid  = [st.valid for st in states]
        if not any(valid):
            continue
        last_idx = max(j for j, v in enumerate(valid) if v)
        st   = states[last_idx]
        dist = ((st.center_x - origin[0])**2 + (st.center_y - origin[1])**2)**0.5
        candidates.append((dist, i, track))

    candidates.sort(key=lambda x: x[0])
    for slot, (_, _, track) in enumerate(candidates[:max_agents]):
        states = track.states[:HIST_STEPS]
        xy     = np.zeros((HIST_STEPS, 2), dtype=np.float32)
        valid  = np.zeros((HIST_STEPS,),   dtype=bool)
        for t, st in enumerate(states):
            if st.valid:
                pt       = np.array([st.center_x, st.center_y], dtype=np.float32)
                xy[t]    = (pt - origin) @ R.T
                valid[t] = True
        social_xy[slot]    = xy
        social_valid[slot] = valid
        social_mask[slot]  = True

    return social_xy, social_valid, social_mask


def extract_full_features(scenario):
    records = []
    for req in scenario.tracks_to_predict:
        track  = scenario.tracks[req.track_index]
        states = track.states

        hist_xy    = np.zeros((HIST_STEPS, 2),   dtype=np.float32)
        hist_hv    = np.zeros((HIST_STEPS, 3),   dtype=np.float32)
        hist_valid = np.zeros((HIST_STEPS,),      dtype=bool)
        fut_xy     = np.zeros((FUTURE_STEPS, 2), dtype=np.float32)
        fut_valid  = np.zeros((FUTURE_STEPS,),    dtype=bool)

        for t in range(HIST_STEPS):
            if t < len(states) and states[t].valid:
                st = states[t]
                hist_xy[t]    = [st.center_x, st.center_y]
                hist_hv[t]    = [st.heading, st.velocity_x, st.velocity_y]
                hist_valid[t] = True

        for t in range(FUTURE_STEPS):
            idx = HIST_STEPS + t
            if idx < len(states) and states[idx].valid:
                st = states[idx]
                fut_xy[t]    = [st.center_x, st.center_y]
                fut_valid[t] = True

        origin       = hist_xy[hist_valid][-1] if hist_valid.any() else np.zeros(2)
        last_heading = hist_hv[hist_valid][-1, 0] if hist_valid.any() else 0.0
        cos_h, sin_h = np.cos(-last_heading), np.sin(-last_heading)
        R = np.array([[cos_h, -sin_h], [sin_h, cos_h]], dtype=np.float32)

        hist_xy_norm              = (hist_xy - origin) @ R.T
        fut_xy_norm               = (fut_xy  - origin) @ R.T
        hist_xy_norm[~hist_valid] = 0.0
        fut_xy_norm[~fut_valid]   = 0.0
        hist_hv_norm              = hist_hv.copy()
        hist_hv_norm[:, 1:]      /= 10.0

        lane_pts, lane_valid, lane_type = extract_map_features(scenario, origin, last_heading)
        soc_xy, soc_valid, soc_mask     = extract_social_agents(scenario, req.track_index, origin, last_heading)

        records.append({
            'history_xy'   : hist_xy_norm,
            'history_hv'   : hist_hv_norm,
            'history_valid': hist_valid,
            'future_xy'    : fut_xy_norm,
            'future_valid' : fut_valid,
            'lane_pts'     : lane_pts,
            'lane_valid'   : lane_valid,
            'lane_type'    : lane_type,
            'social_xy'    : soc_xy,
            'social_valid' : soc_valid,
            'social_mask'  : soc_mask,
            'agent_type'   : AGENT_TYPE_INT.get(track.object_type, 3),
            'scenario_id'  : scenario.scenario_id,
        })
    return records


def slice_close_interaction(scenario, threshold_m=5.0):
    pred_tracks = [scenario.tracks[t.track_index] for t in scenario.tracks_to_predict]
    if len(pred_tracks) < 2:
        return False
    for i in range(len(pred_tracks)):
        for j in range(i + 1, len(pred_tracks)):
            si, sj = pred_tracks[i].states, pred_tracks[j].states
            for k in range(min(len(si), len(sj))):
                if si[k].valid and sj[k].valid:
                    dx = si[k].center_x - sj[k].center_x
                    dy = si[k].center_y - sj[k].center_y
                    if (dx*dx + dy*dy)**0.5 < threshold_m:
                        return True
    return False


def slice_high_curvature(scenario, radius_thresh_m=20.0):
    for req in scenario.tracks_to_predict:
        track  = scenario.tracks[req.track_index]
        states = track.states[:HIST_STEPS]
        for k in range(1, len(states) - 1):
            if not (states[k-1].valid and states[k].valid and states[k+1].valid):
                continue
            speed = (states[k].velocity_x**2 + states[k].velocity_y**2)**0.5
            if speed < 1.0:
                continue
            dh = states[k].heading - states[k-1].heading
            dh = (dh + np.pi) % (2 * np.pi) - np.pi
            if abs(dh) / 0.1 > 1e-3 and speed / (abs(dh) / 0.1) < radius_thresh_m:
                return True
    return False


print('extractors ready')


## Process shards and save features to GCS

Skip this cell if `features/map_v1/` already exists in your bucket.

In [ ]:
N_SHARDS      = 50
MAX_SCENARIOS = 200

print(f'processing {N_SHARDS} shards...')
for i, shard_blob in enumerate(tqdm(val_shards[:N_SHARDS], desc='shards')):
    shard_path   = f'gs://{WOMD_BUCKET}/{shard_blob.name}'
    dataset      = tf.data.TFRecordDataset(shard_path, compression_type='')
    shard_feats  = []
    shard_labels = []

    for j, raw in enumerate(dataset):
        if j >= MAX_SCENARIOS:
            break
        sc = scenario_pb2.Scenario()
        sc.ParseFromString(raw.numpy())
        is_close = slice_close_interaction(sc)
        is_curv  = slice_high_curvature(sc)
        for f in extract_full_features(sc):
            shard_feats.append(f)
            shard_labels.append({'close_int': is_close, 'high_curv': is_curv})

    if not shard_feats:
        continue

    arrays = {
        'history_xy'     : np.stack([f['history_xy']    for f in shard_feats]),
        'history_hv'     : np.stack([f['history_hv']    for f in shard_feats]),
        'history_valid'  : np.stack([f['history_valid'] for f in shard_feats]),
        'future_xy'      : np.stack([f['future_xy']     for f in shard_feats]),
        'future_valid'   : np.stack([f['future_valid']  for f in shard_feats]),
        'lane_pts'       : np.stack([f['lane_pts']      for f in shard_feats]),
        'lane_valid'     : np.stack([f['lane_valid']    for f in shard_feats]),
        'lane_type'      : np.stack([f['lane_type']     for f in shard_feats]),
        'social_xy'      : np.stack([f['social_xy']     for f in shard_feats]),
        'social_valid'   : np.stack([f['social_valid']  for f in shard_feats]),
        'social_mask'    : np.stack([f['social_mask']   for f in shard_feats]),
        'agent_type'     : np.array([f['agent_type']    for f in shard_feats]),
        'label_close_int': np.array([l['close_int']     for l in shard_labels]),
        'label_high_curv': np.array([l['high_curv']     for l in shard_labels]),
    }
    buf = io.BytesIO()
    np.savez_compressed(buf, **arrays)
    buf.seek(0)
    bucket.blob(f'features/map_v1/shard_{i:04d}.npz').upload_from_file(
        buf, content_type='application/octet-stream')

print(f'done — saved to gs://{GCS_BUCKET}/features/map_v1/')


## Model architecture

MTR-Lite: agent GRU encoder, PointNet lane encoder, cross-attention over map,
social mean-pool encoder, fused context decoded into K=6 trajectory modes.

In [ ]:
class MapAwareDataset(Dataset):
    def __init__(self, data):
        self.hist_xy     = torch.tensor(data['history_xy'],    dtype=torch.float32)
        self.hist_hv     = torch.tensor(data['history_hv'],    dtype=torch.float32)
        self.hist_valid  = torch.tensor(data['history_valid'], dtype=torch.float32)
        self.fut_xy      = torch.tensor(data['future_xy'],     dtype=torch.float32)
        self.fut_valid   = torch.tensor(data['future_valid'],  dtype=torch.float32)
        self.lane_pts    = torch.tensor(data['lane_pts'],      dtype=torch.float32)
        self.lane_valid  = torch.tensor(data['lane_valid'],    dtype=torch.float32)
        self.social_xy   = torch.tensor(data['social_xy'],     dtype=torch.float32)
        self.social_mask = torch.tensor(data['social_mask'],   dtype=torch.float32)

    def __len__(self):
        return len(self.hist_xy)

    def __getitem__(self, i):
        return (self.hist_xy[i], self.hist_hv[i], self.hist_valid[i],
                self.lane_pts[i], self.lane_valid[i],
                self.social_xy[i], self.social_mask[i],
                self.fut_xy[i], self.fut_valid[i])


class AgentEncoder(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.gru = nn.GRU(5, hidden, num_layers=2, batch_first=True, dropout=0.1)

    def forward(self, hxy, hhv):
        _, h = self.gru(torch.cat([hxy, hhv], dim=-1))
        return h[-1]


class LaneEncoder(nn.Module):
    def __init__(self, out_dim=64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(2, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, out_dim),
        )
        self.out_dim = out_dim

    def forward(self, lane_pts, lane_valid):
        B, L, P, _ = lane_pts.shape
        feat = self.mlp(lane_pts.reshape(B * L, P, 2)).max(dim=1).values.reshape(B, L, self.out_dim)
        return feat * lane_valid.unsqueeze(-1)


class SocialEncoder(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.gru    = nn.GRU(2, hidden, num_layers=1, batch_first=True)
        self.hidden = hidden

    def forward(self, social_xy, social_mask):
        B, A, T, _ = social_xy.shape
        _, h = self.gru(social_xy.reshape(B * A, T, 2))
        h    = h[-1].reshape(B, A, self.hidden) * social_mask.unsqueeze(-1)
        return h.sum(dim=1) / social_mask.sum(dim=1, keepdim=True).clamp(min=1)


class CrossAttention(nn.Module):
    def __init__(self, agent_dim, lane_dim, out_dim, n_heads=4):
        super().__init__()
        self.q_proj = nn.Linear(agent_dim, out_dim)
        self.k_proj = nn.Linear(lane_dim,  out_dim)
        self.v_proj = nn.Linear(lane_dim,  out_dim)
        self.attn   = nn.MultiheadAttention(out_dim, n_heads, batch_first=True, dropout=0.1)
        self.norm   = nn.LayerNorm(out_dim)

    def forward(self, agent_feat, lane_feat, lane_valid):
        q      = self.q_proj(agent_feat).unsqueeze(1)
        out, _ = self.attn(q, self.k_proj(lane_feat), self.v_proj(lane_feat),
                           key_padding_mask=(lane_valid == 0))
        return self.norm(out.squeeze(1))


class MultiModalDecoder(nn.Module):
    def __init__(self, ctx_dim, K=6, future_steps=80):
        super().__init__()
        self.K            = K
        self.future_steps = future_steps
        self.traj_heads   = nn.ModuleList([
            nn.Sequential(
                nn.Linear(ctx_dim, 256), nn.ReLU(),
                nn.Linear(256, 256),     nn.ReLU(),
                nn.Linear(256, future_steps * 2),
            ) for _ in range(K)
        ])
        self.score_head = nn.Linear(ctx_dim, K)

    def forward(self, ctx):
        trajs  = torch.stack([h(ctx).reshape(-1, self.future_steps, 2) for h in self.traj_heads], dim=1)
        return trajs, self.score_head(ctx)


class MTRLite(nn.Module):
    def __init__(self, K=6, future_steps=80):
        super().__init__()
        self.agent_enc  = AgentEncoder(hidden=128)
        self.lane_enc   = LaneEncoder(out_dim=64)
        self.social_enc = SocialEncoder(hidden=64)
        self.cross_attn = CrossAttention(128, 64, 128, n_heads=4)
        self.decoder    = MultiModalDecoder(192, K=K, future_steps=future_steps)
        self.K          = K

    def forward(self, hxy, hhv, hval, lpts, lval, sxy, smask):
        agent_feat = self.agent_enc(hxy, hhv)
        lane_feat  = self.lane_enc(lpts, lval)
        map_ctx    = self.cross_attn(agent_feat, lane_feat, lval)
        social_ctx = self.social_enc(sxy, smask)
        ctx        = torch.cat([map_ctx, social_ctx], dim=-1)
        return self.decoder(ctx)


model    = MTRLite(K=6, future_steps=80).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'MTR-Lite ready  params={n_params:,}  K=6  device={device}')


## Load features from GCS

In [ ]:
print('loading features from GCS...')
all_data = []
for b in sorted(bucket.list_blobs(prefix='features/map_v1/'), key=lambda x: x.name):
    buf = io.BytesIO(b.download_as_bytes())
    all_data.append(dict(np.load(buf, allow_pickle=True)))

data = {k: np.concatenate([d[k] for d in all_data]) for k in all_data[0]}
N    = len(data['history_xy'])

close_mask = data['label_close_int'].astype(bool)
curv_mask  = data['label_high_curv'].astype(bool)
both_mask  = close_mask & curv_mask

print(f'loaded {N:,} agent-samples')
print(f'  history_xy  : {data["history_xy"].shape}')
print(f'  lane_pts    : {data["lane_pts"].shape}')
print(f'  social_xy   : {data["social_xy"].shape}')
print(f'  close-int   : {close_mask.sum():,} ({100*close_mask.mean():.1f}%)')
print(f'  high-curv   : {curv_mask.sum():,} ({100*curv_mask.mean():.1f}%)')
print(f'  both        : {both_mask.sum():,} ({100*both_mask.mean():.1f}%)')

dataset = MapAwareDataset(data)
loader  = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
print(f'  batches/epoch : {len(loader)}')


## Training

Winner-takes-all loss: regression on the closest mode plus cross-entropy to score it highest.
Cosine LR schedule, gradient clipping at 1.0, best checkpoint saved to GCS.

In [ ]:
def wta_loss(trajs, scores, fut_xy, fut_valid):
    B, K, T, _ = trajs.shape
    gt       = fut_xy.unsqueeze(1).expand_as(trajs)
    msk      = fut_valid.unsqueeze(1).expand(B, K, T)
    dist     = torch.norm(trajs - gt, dim=-1)
    mode_ade = (dist * msk).sum(-1) / (msk.sum(-1) + 1e-6)
    best_idx = mode_ade.argmin(dim=1)

    winner_trajs = trajs[torch.arange(B), best_idx]
    reg_loss = (torch.norm(winner_trajs - fut_xy, dim=-1) * fut_valid).sum() / (fut_valid.sum() + 1e-6)
    cls_loss = F.cross_entropy(scores, best_idx)

    return reg_loss + 0.5 * cls_loss, mode_ade.min(dim=1).values.mean()


N_EPOCHS    = 50
optimizer   = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
best_minade = float('inf')

print(f'training for {N_EPOCHS} epochs')
print(f'{"epoch":>6} {"loss":>8} {"minADE":>8} {"lr":>10}')
print('-' * 36)

for epoch in range(N_EPOCHS):
    model.train()
    total_loss = 0
    total_ade  = 0

    for batch in loader:
        hxy, hhv, hval, lpts, lval, sxy, smask, fxy, fval = [x.to(device) for x in batch]
        trajs, scores = model(hxy, hhv, hval, lpts, lval, sxy, smask)
        loss, minade  = wta_loss(trajs, scores, fxy, fval)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_ade  += minade.item()

    scheduler.step()
    avg_loss   = total_loss / len(loader)
    avg_minade = total_ade  / len(loader)

    if avg_minade < best_minade:
        best_minade = avg_minade
        torch.save(model.state_dict(), '/tmp/mtr_lite_best.pt')

    if (epoch + 1) % 5 == 0:
        lr = scheduler.get_last_lr()[0]
        print(f'{epoch+1:>6} {avg_loss:>8.4f} {avg_minade:>8.4f} {lr:>10.2e}')

bucket.blob('checkpoints/mtr_lite_50ep.pt').upload_from_filename('/tmp/mtr_lite_best.pt')
print(f'training done  best minADE@6={best_minade:.4f}m')
print(f'checkpoint saved to gs://{GCS_BUCKET}/checkpoints/mtr_lite_50ep.pt')


## Evaluation across tail slices

In [ ]:
def evaluate_mtr(model, data, device, slice_mask=None):
    loader = DataLoader(MapAwareDataset(data), batch_size=512, shuffle=False, num_workers=2)

    all_min_ade = []
    all_min_fde = []

    model.eval()
    with torch.no_grad():
        for batch in loader:
            hxy, hhv, hval, lpts, lval, sxy, smask, fxy, fval = [x.to(device) for x in batch]
            trajs, scores = model(hxy, hhv, hval, lpts, lval, sxy, smask)
            B, K, T, _   = trajs.shape

            fxy_np   = fxy.cpu().numpy()
            fval_np  = fval.cpu().numpy().astype(bool)
            trajs_np = trajs.cpu().numpy()

            for i in range(B):
                valid = fval_np[i]
                if not valid.any():
                    continue
                gt      = fxy_np[i]
                ades    = []
                fdes    = []
                for k in range(K):
                    diff = np.linalg.norm(trajs_np[i, k] - gt, axis=1)
                    ades.append(diff[valid].mean())
                    fdes.append(diff[np.where(valid)[0][-1]])
                all_min_ade.append(min(ades))
                all_min_fde.append(min(fdes))

    all_min_ade = np.array(all_min_ade)
    all_min_fde = np.array(all_min_fde)

    if slice_mask is not None:
        all_min_ade = all_min_ade[slice_mask[:len(all_min_ade)]]
        all_min_fde = all_min_fde[slice_mask[:len(all_min_fde)]]

    miss_rate = (all_min_fde > 2.0).mean() * 100
    return {
        'minADE@6': round(float(all_min_ade.mean()), 3),
        'minFDE@6': round(float(all_min_fde.mean()), 3),
        'MissRate': round(float(miss_rate), 1),
        'N'       : len(all_min_ade),
    }


slices = {
    'Aggregate'         : None,
    'Close-interaction' : close_mask,
    'High-curvature'    : curv_mask,
    'Both (hardest)'    : both_mask,
}

print(f'{"Slice":<22} {"minADE@6":>9} {"minFDE@6":>9} {"MissRate":>9} {"N":>6}')
print('=' * 60)

mtr_results = {}
for sname, mask in slices.items():
    m = evaluate_mtr(model, data, device, mask)
    mtr_results[sname] = m
    print(f'{sname:<22} {m["minADE@6"]:>9.3f} {m["minFDE@6"]:>9.3f} {m["MissRate"]:>8.1f}% {m["N"]:>6}')


## Comparison plot — baseline progression

In [ ]:
old_results = {
    'Aggregate'         : {'CV': 9.35, 'MLP': 5.40, 'GRU': 5.28},
    'Close-interaction' : {'CV': 8.98, 'MLP': 5.36, 'GRU': 5.27},
    'High-curvature'    : {'CV': 9.13, 'MLP': 5.27, 'GRU': 5.17},
    'Both (hardest)'    : {'CV': 8.75, 'MLP': 5.23, 'GRU': 5.14},
}

slice_names = list(old_results.keys())
x      = np.arange(len(slice_names))
width  = 0.18
colors = ['#B0BEC5', '#90CAF9', '#4C9BE8', '#5BBF8A']

fig, ax = plt.subplots(figsize=(13, 6))
ax.set_title('WOMD model progression (ADE / minADE@6, meters)', fontsize=13, fontweight='bold')

for i, (model_name, color) in enumerate(zip(['CV', 'MLP', 'GRU (baseline)', 'MTR-Lite (ours)'], colors)):
    if model_name == 'MTR-Lite (ours)':
        vals  = [mtr_results[s]['minADE@6'] for s in slice_names]
        label = 'MTR-Lite  minADE@6'
    else:
        key   = model_name.split(' ')[0]
        vals  = [old_results[s][key] for s in slice_names]
        label = f'{model_name}  ADE@1'

    bars = ax.bar(x + i * width, vals, width, label=label, color=color, edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                f'{val:.2f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([s.replace(' ', '\n') for s in slice_names], fontsize=9)
ax.set_ylabel('ADE (meters)')
ax.legend(fontsize=9, loc='upper right')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 11)

plt.tight_layout()
plt.savefig('/tmp/mtr_progression.png', dpi=150, bbox_inches='tight')
bucket.blob('plots/mtr_progression.png').upload_from_filename('/tmp/mtr_progression.png')
plt.show()

print('\nfinal summary')
print(f'{"model":<22} {"aggregate":>12} {"both (hardest)":>15}')
print('-' * 52)
for name, key in [('CV', 'CV'), ('MLP', 'MLP'), ('GRU', 'GRU')]:
    print(f'{name:<22} {old_results["Aggregate"][key]:>10.2f}m {old_results["Both (hardest)"][key]:>13.2f}m')

mtr_agg  = mtr_results['Aggregate']['minADE@6']
mtr_both = mtr_results['Both (hardest)']['minADE@6']
print(f'{"MTR-Lite":<22} {mtr_agg:>10.2f}m {mtr_both:>13.2f}m')
print(f'\nimprovement over GRU: {((5.28 - mtr_agg) / 5.28 * 100):.1f}% aggregate')
print(f'improvement over GRU: {((5.14 - mtr_both) / 5.14 * 100):.1f}% hardest tail')


## Visualization

Find representative scenarios of each type, run inference, animate predictions over the HD map.

In [ ]:
scenario_bank = {
    'close_interaction'  : None,
    'high_curvature'     : None,
    'pedestrian_crossing': None,
    'multi_agent'        : None,
}

for shard_blob in val_shards[1:10]:
    shard_path = f'gs://{WOMD_BUCKET}/{shard_blob.name}'
    dataset    = tf.data.TFRecordDataset(shard_path, compression_type='')

    for raw in dataset:
        sc = scenario_pb2.Scenario()
        sc.ParseFromString(raw.numpy())
        pred_tracks = [sc.tracks[t.track_index] for t in sc.tracks_to_predict]

        if scenario_bank['close_interaction'] is None:
            for i in range(len(pred_tracks)):
                for j in range(i + 1, len(pred_tracks)):
                    si, sj = pred_tracks[i].states, pred_tracks[j].states
                    for k in range(min(len(si), len(sj))):
                        if si[k].valid and sj[k].valid:
                            dx = si[k].center_x - sj[k].center_x
                            dy = si[k].center_y - sj[k].center_y
                            if (dx*dx + dy*dy)**0.5 < 3.0:
                                scenario_bank['close_interaction'] = sc
                                break

        if scenario_bank['high_curvature'] is None:
            if slice_high_curvature(sc, radius_thresh_m=10.0):
                scenario_bank['high_curvature'] = sc

        if scenario_bank['pedestrian_crossing'] is None:
            for req in sc.tracks_to_predict:
                if sc.tracks[req.track_index].object_type == scenario_pb2.Track.TYPE_PEDESTRIAN:
                    scenario_bank['pedestrian_crossing'] = sc
                    break

        if scenario_bank['multi_agent'] is None:
            if len(sc.tracks_to_predict) >= 4:
                scenario_bank['multi_agent'] = sc

        if all(v is not None for v in scenario_bank.values()):
            break

    if all(v is not None for v in scenario_bank.values()):
        break

for name, sc in scenario_bank.items():
    if sc:
        types = [scenario_pb2.Track.ObjectType.Name(sc.tracks[r.track_index].object_type)
                 for r in sc.tracks_to_predict]
        print(f'{name:<28} id={sc.scenario_id[:16]}  agents={len(sc.tracks_to_predict)}  types={types}')
    else:
        print(f'{name:<28} not found')


In [ ]:
# change PICK to switch scenario type
# options: 'close_interaction' | 'high_curvature' | 'pedestrian_crossing' | 'multi_agent'
PICK = 'multi_agent'

target_scenario = scenario_bank[PICK]
print(f'scenario: {target_scenario.scenario_id}')
print(f'agents to predict: {len(target_scenario.tracks_to_predict)}')


def get_model_predictions(scenario, model, device):
    all_preds = []
    model.eval()
    with torch.no_grad():
        for req in scenario.tracks_to_predict:
            track  = scenario.tracks[req.track_index]
            states = track.states

            hist_xy    = np.zeros((HIST_STEPS, 2), dtype=np.float32)
            hist_hv    = np.zeros((HIST_STEPS, 3), dtype=np.float32)
            hist_valid = np.zeros((HIST_STEPS,),   dtype=bool)

            for t in range(HIST_STEPS):
                if t < len(states) and states[t].valid:
                    st = states[t]
                    hist_xy[t]    = [st.center_x, st.center_y]
                    hist_hv[t]    = [st.heading, st.velocity_x, st.velocity_y]
                    hist_valid[t] = True

            origin       = hist_xy[hist_valid][-1] if hist_valid.any() else np.zeros(2)
            last_heading = hist_hv[hist_valid][-1, 0] if hist_valid.any() else 0.0
            cos_h, sin_h = np.cos(-last_heading), np.sin(-last_heading)
            R = np.array([[cos_h, -sin_h], [sin_h, cos_h]], dtype=np.float32)

            hist_xy_norm              = (hist_xy - origin) @ R.T
            hist_hv_norm              = hist_hv.copy()
            hist_hv_norm[:, 1:]      /= 10.0
            hist_xy_norm[~hist_valid] = 0.0

            lane_pts, lane_valid, _ = extract_map_features(scenario, origin, last_heading)
            soc_xy, soc_valid, soc_mask = extract_social_agents(scenario, req.track_index, origin, last_heading)

            def t_(x):
                return torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)

            trajs, scores = model(
                t_(hist_xy_norm), t_(hist_hv_norm),
                t_(hist_valid.astype(np.float32)),
                t_(lane_pts), t_(lane_valid.astype(np.float32)),
                t_(soc_xy), t_(soc_mask.astype(np.float32))
            )

            trajs_np    = trajs[0].detach().cpu().numpy()
            scores_np   = torch.softmax(scores[0], dim=0).detach().cpu().numpy()
            trajs_world = np.stack([trajs_np[k] @ R.T + origin for k in range(trajs_np.shape[0])])

            all_preds.append({
                'track_idx': req.track_index,
                'trajs'    : trajs_world,
                'scores'   : scores_np,
            })
    return all_preds


preds = get_model_predictions(target_scenario, model, device)
print(f'predictions for {len(preds)} agents')
for p in preds:
    best_k = p['scores'].argmax()
    print(f'  agent {p["track_idx"]}  best_mode={best_k}  score={p["scores"][best_k]:.3f}')


## Inline animation

In [ ]:
AGENT_COLORS = {0: '#4C9BE8', 1: '#F28C38', 2: '#5BBF8A', 3: '#BBBBBB'}
PRED_COLORS  = ['#FF4444', '#FF8800', '#FFCC00', '#88FF44', '#44FFCC', '#AA44FF']

focal_pos = [
    (st.center_x, st.center_y)
    for req in target_scenario.tracks_to_predict
    for st in target_scenario.tracks[req.track_index].states[:HIST_STEPS]
    if st.valid
]
cx = np.mean([p[0] for p in focal_pos])
cy = np.mean([p[1] for p in focal_pos])

VIEW        = 60
TOTAL       = min(91, len(target_scenario.timestamps_seconds))
predict_ids = {req.track_index for req in target_scenario.tracks_to_predict}


def draw_map(ax, scenario):
    for feat in scenario.map_features:
        if feat.HasField('lane'):
            pts = np.array([(p.x, p.y) for p in feat.lane.polyline])
            if len(pts) > 1:
                ax.plot(pts[:, 0], pts[:, 1], color='#444444', linewidth=0.4, alpha=0.5, zorder=1)
        elif feat.HasField('road_edge'):
            pts = np.array([(p.x, p.y) for p in feat.road_edge.polyline])
            if len(pts) > 1:
                ax.plot(pts[:, 0], pts[:, 1], color='#888888', linewidth=0.8, alpha=0.6, zorder=1)
        elif feat.HasField('crosswalk'):
            pts = np.array([(p.x, p.y) for p in feat.crosswalk.polygon])
            if len(pts) > 1:
                ax.add_patch(plt.Polygon(pts, fill=True, facecolor='#3a3a3a',
                             edgecolor='#666666', linewidth=0.5, alpha=0.4, zorder=1))


fig, ax = plt.subplots(figsize=(10, 10), facecolor='#1a1a1a')


def animate(frame_t):
    ax.clear()
    ax.set_facecolor('#1a1a1a')
    ax.set_xlim(cx - VIEW, cx + VIEW)
    ax.set_ylim(cy - VIEW, cy + VIEW)
    ax.set_aspect('equal')
    ax.axis('off')
    draw_map(ax, target_scenario)

    for i, track in enumerate(target_scenario.tracks):
        color    = AGENT_COLORS.get(track.object_type, '#BBBBBB')
        is_focal = i in predict_ids
        trail_x, trail_y = [], []
        for t2 in range(max(0, frame_t - 10), frame_t + 1):
            if t2 < len(track.states) and track.states[t2].valid:
                trail_x.append(track.states[t2].center_x)
                trail_y.append(track.states[t2].center_y)
        if trail_x:
            ax.plot(trail_x, trail_y, color=color,
                    linewidth=2.0 if is_focal else 0.8,
                    alpha=0.9 if is_focal else 0.3, zorder=3)
        if frame_t < len(track.states) and track.states[frame_t].valid:
            st = track.states[frame_t]
            ax.plot(st.center_x, st.center_y, 'o', color=color,
                    markersize=10 if is_focal else 4, zorder=5,
                    markeredgecolor='white',
                    markeredgewidth=0.8 if is_focal else 0.2)

    if frame_t >= HIST_STEPS:
        steps_shown = min(frame_t - HIST_STEPS + 1, 80)
        for p in preds:
            for rank, k in enumerate(np.argsort(p['scores'])[::-1][:3]):
                pts = p['trajs'][k, :steps_shown]
                ax.plot(pts[:, 0], pts[:, 1], '--',
                        color=PRED_COLORS[k % len(PRED_COLORS)],
                        linewidth=[2.5, 1.5, 1.0][rank],
                        alpha=[0.9, 0.5, 0.25][rank], zorder=6)
                if rank == 0 and steps_shown > 1:
                    ax.plot(pts[-1, 0], pts[-1, 1], '*',
                            color=PRED_COLORS[k], markersize=14, zorder=7)

    phase = 'history' if frame_t < HIST_STEPS else f'predicting t+{frame_t - HIST_STEPS + 1}'
    ax.set_title(f'{target_scenario.scenario_id[:16]}   t={frame_t * 0.1:.1f}s  [{phase}]',
                 color='white', fontsize=10, pad=6)
    ax.legend(handles=[
        plt.Line2D([0], [0], color='#4C9BE8', lw=2, label='vehicle (focal)'),
        plt.Line2D([0], [0], color='#BBBBBB', lw=1, alpha=0.4, label='other agents'),
        plt.Line2D([0], [0], color='#FF4444', lw=2, ls='--', label='best prediction'),
        plt.Line2D([0], [0], color='#FF8800', lw=1.5, ls='--', alpha=0.5, label='alt modes'),
    ], loc='upper right', facecolor='#2a2a2a', edgecolor='#555', labelcolor='white', fontsize=8)


anim = FuncAnimation(fig, animate, frames=TOTAL, interval=100, repeat=True)
plt.close()
HTML(anim.to_jshtml())


## Export MP4 videos

Renders three scenario types and saves as downloadable MP4 files.

In [ ]:
!apt-get install -qq ffmpeg

scenarios_to_render = {
    'intersection_multiagent': None,
    'pedestrian_vehicle'     : None,
    'highway_curvature'      : None,
}

print('scanning for render candidates...')
for shard_blob in val_shards[2:15]:
    shard_path = f'gs://{WOMD_BUCKET}/{shard_blob.name}'
    dataset    = tf.data.TFRecordDataset(shard_path, compression_type='')

    for raw in dataset:
        sc          = scenario_pb2.Scenario()
        sc.ParseFromString(raw.numpy())
        pred_tracks = [sc.tracks[t.track_index] for t in sc.tracks_to_predict]
        agent_types = [sc.tracks[t.track_index].object_type for t in sc.tracks_to_predict]

        if scenarios_to_render['intersection_multiagent'] is None and len(pred_tracks) >= 4:
            positions = []
            for t in sc.tracks_to_predict:
                for st in sc.tracks[t.track_index].states[:11]:
                    if st.valid:
                        positions.append((st.center_x, st.center_y))
                        break
            if len(positions) >= 4:
                xs     = [p[0] for p in positions]
                ys     = [p[1] for p in positions]
                spread = ((max(xs) - min(xs))**2 + (max(ys) - min(ys))**2)**0.5
                if spread > 15:
                    scenarios_to_render['intersection_multiagent'] = sc

        if scenarios_to_render['pedestrian_vehicle'] is None:
            has_ped = scenario_pb2.Track.TYPE_PEDESTRIAN in agent_types
            has_veh = scenario_pb2.Track.TYPE_VEHICLE in agent_types
            if has_ped and has_veh:
                ped_tracks = [sc.tracks[sc.tracks_to_predict[i].track_index]
                              for i, t in enumerate(agent_types) if t == scenario_pb2.Track.TYPE_PEDESTRIAN]
                veh_tracks = [sc.tracks[sc.tracks_to_predict[i].track_index]
                              for i, t in enumerate(agent_types) if t == scenario_pb2.Track.TYPE_VEHICLE]
                for pt in ped_tracks:
                    for vt in veh_tracks:
                        for k in range(min(len(pt.states), len(vt.states), 50)):
                            if pt.states[k].valid and vt.states[k].valid:
                                dx = pt.states[k].center_x - vt.states[k].center_x
                                dy = pt.states[k].center_y - vt.states[k].center_y
                                if (dx*dx + dy*dy)**0.5 < 8.0:
                                    scenarios_to_render['pedestrian_vehicle'] = sc
                                    break

        if scenarios_to_render['highway_curvature'] is None and slice_high_curvature(sc, radius_thresh_m=12.0):
            for req in sc.tracks_to_predict:
                speeds = [(st.velocity_x**2 + st.velocity_y**2)**0.5
                          for st in sc.tracks[req.track_index].states[:11] if st.valid]
                if speeds and max(speeds) > 8.0:
                    scenarios_to_render['highway_curvature'] = sc
                    break

        if all(v is not None for v in scenarios_to_render.values()):
            break
    if all(v is not None for v in scenarios_to_render.values()):
        break


def render_scenario_video(scenario, preds, output_path, title_label, view_range=70, fps=10, dpi=120):
    focal_pos = [
        (st.center_x, st.center_y)
        for req in scenario.tracks_to_predict
        for st in scenario.tracks[req.track_index].states[:11]
        if st.valid
    ]
    cx = np.mean([p[0] for p in focal_pos])
    cy = np.mean([p[1] for p in focal_pos])

    TOTAL       = min(91, len(scenario.timestamps_seconds))
    predict_ids = {req.track_index for req in scenario.tracks_to_predict}
    AC          = {0: '#4C9BE8', 1: '#F28C38', 2: '#5BBF8A', 3: '#BBBBBB'}
    PC          = ['#FF4444', '#FF8800', '#FFDD00', '#44FF88', '#44CCFF', '#CC44FF']

    fig, ax = plt.subplots(figsize=(9, 9), facecolor='#111111')
    fig.subplots_adjust(left=0, right=1, top=0.93, bottom=0)

    def draw(ax):
        for feat in scenario.map_features:
            if feat.HasField('lane'):
                pts = np.array([(p.x, p.y) for p in feat.lane.polyline])
                if len(pts) > 1:
                    ax.plot(pts[:, 0], pts[:, 1], color='#3a3a3a', linewidth=0.6, zorder=1)
            elif feat.HasField('road_edge'):
                pts = np.array([(p.x, p.y) for p in feat.road_edge.polyline])
                if len(pts) > 1:
                    ax.plot(pts[:, 0], pts[:, 1], color='#666666', linewidth=1.0, zorder=2)
            elif feat.HasField('crosswalk'):
                pts = np.array([(p.x, p.y) for p in feat.crosswalk.polygon])
                if len(pts) > 2:
                    ax.add_patch(plt.Polygon(pts, fill=True, facecolor='#2a2a2a',
                                 edgecolor='#555555', linewidth=0.5, alpha=0.6, zorder=1))

    def frame(t):
        ax.clear()
        ax.set_facecolor('#111111')
        ax.set_xlim(cx - view_range, cx + view_range)
        ax.set_ylim(cy - view_range, cy + view_range)
        ax.set_aspect('equal')
        ax.axis('off')
        draw(ax)

        for i, track in enumerate(scenario.tracks):
            is_focal = i in predict_ids
            color    = AC.get(track.object_type, '#BBBBBB')
            trail_x, trail_y = [], []
            for t2 in range(max(0, t - 15), t + 1):
                if t2 < len(track.states) and track.states[t2].valid:
                    trail_x.append(track.states[t2].center_x)
                    trail_y.append(track.states[t2].center_y)
            if trail_x:
                ax.plot(trail_x, trail_y, color=color,
                        linewidth=2.5 if is_focal else 0.8,
                        alpha=0.95 if is_focal else 0.25,
                        solid_capstyle='round', zorder=3)
            if t < len(track.states) and track.states[t].valid:
                st = track.states[t]
                ax.plot(st.center_x, st.center_y, 'o', color=color,
                        markersize=11 if is_focal else 4,
                        markeredgecolor='white',
                        markeredgewidth=1.0 if is_focal else 0.2,
                        zorder=5 if is_focal else 4)

        if t >= HIST_STEPS:
            steps = min(t - HIST_STEPS + 1, 80)
            for p in preds:
                for rank, k in enumerate(np.argsort(p['scores'])[::-1][:3]):
                    pts = p['trajs'][k, :steps]
                    if len(pts) < 2:
                        continue
                    ax.plot(pts[:, 0], pts[:, 1], '--',
                            color=PC[k % len(PC)],
                            linewidth=[3.0, 1.8, 1.0][rank],
                            alpha=[0.95, 0.5, 0.25][rank],
                            solid_capstyle='round', zorder=6)
                    if rank == 0:
                        ax.plot(pts[-1, 0], pts[-1, 1], '*', color=PC[k],
                                markersize=16, zorder=7, markeredgecolor='white', markeredgewidth=0.5)

        phase = 'observing' if t < HIST_STEPS else f'predicting (+{(t - HIST_STEPS) * 0.1:.1f}s)'
        ax.set_title(f'MTR-Lite  |  {title_label}  |  t={t * 0.1:.1f}s  [{phase}]',
                     color='white', fontsize=11, pad=6, fontweight='bold', loc='center')
        ax.legend(handles=[
            mpatches.Patch(color='#4C9BE8', label='vehicle'),
            mpatches.Patch(color='#F28C38', label='pedestrian'),
            mpatches.Patch(color='#5BBF8A', label='cyclist'),
            plt.Line2D([0], [0], color='#FF4444', lw=2, ls='--', label='best prediction'),
            plt.Line2D([0], [0], color='#FF8800', lw=1.5, ls='--', alpha=0.5, label='alt modes'),
        ], loc='upper right', facecolor='#1e1e1e', edgecolor='#444',
           labelcolor='white', fontsize=8.5, framealpha=0.85)

    anim   = FuncAnimation(fig, frame, frames=TOTAL, interval=100)
    writer = FFMpegWriter(fps=fps, metadata={'title': title_label},
                          extra_args=['-vcodec', 'libx264', '-pix_fmt', 'yuv420p'])
    anim.save(output_path, writer=writer, dpi=dpi, savefig_kwargs={'facecolor': '#111111'})
    plt.close()
    print(f'saved {output_path}  ({os.path.getsize(output_path) // 1024} KB)')
    return output_path


video_configs = [
    ('intersection_multiagent', 'Busy Intersection', 65),
    ('pedestrian_vehicle',      'Pedestrian x Vehicle', 50),
    ('highway_curvature',       'High-Speed Curve', 80),
]

output_files = []
for sc_key, label, view in video_configs:
    sc = scenarios_to_render[sc_key]
    if sc is None:
        print(f'skipping {sc_key} — not found')
        continue
    print(f'rendering: {label}...')
    p    = get_model_predictions(sc, model, device)
    path = f'/tmp/{sc_key}.mp4'
    render_scenario_video(sc, p, path, label, view_range=view)
    output_files.append(path)

print(f'\n{len(output_files)} videos ready')
for path in output_files:
    name = os.path.basename(path)
    size = os.path.getsize(path) // 1024
    display(FileLink(path, result_html_prefix=f'{name} ({size} KB)   '))
